# Real-World MCP Server Project: Customer Support Assistant

This notebook builds a practical customer support MCP server with multiple tools, then connects it to OpenAI so a GPT model can handle real support conversations using your custom backend.

## What We're Building

A customer support MCP server that an AI assistant can use to:
- **Look up orders** by order ID or customer email
- **Check product availability** and pricing
- **Process return requests** for eligible orders
- **Search the FAQ** for common questions

This mirrors a real-world pattern: your MCP server wraps your internal APIs and databases, and any MCP-compatible AI client can use them.

### Architecture

```
User Question → OpenAI Responses API → MCP Server → Your Backend (simulated)
                                          ↓
                                    Tool calls:
                                    - lookup_order
                                    - search_orders_by_email
                                    - check_product_availability
                                    - request_return
                                    - search_faq
```

## Setup

In [ ]:
%pip install -q fastmcp openai pyngrok

In [ ]:
import os
import json
import time
import threading
from datetime import datetime, timedelta
from getpass import getpass
from openai import OpenAI
from fastmcp import FastMCP

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-4o"

## Part 1: Building the Customer Support Server

We will build the server step by step, starting with the simulated backend data and then adding tools one at a time.

In [ ]:
support_server = FastMCP("Customer Support")

# ============================================================
# Simulated backend data (in production, these would be API calls
# to your real databases and services)
# ============================================================

ORDERS = {
    "ORD-1001": {
        "customer_email": "alice@example.com",
        "items": [{"name": "Wireless Headphones", "qty": 1, "price": 79.99}],
        "total": 79.99,
        "status": "delivered",
        "order_date": "2026-01-15",
        "delivery_date": "2026-01-20",
    },
    "ORD-1002": {
        "customer_email": "alice@example.com",
        "items": [
            {"name": "USB-C Cable", "qty": 2, "price": 12.99},
            {"name": "Phone Case", "qty": 1, "price": 24.99},
        ],
        "total": 50.97,
        "status": "shipped",
        "order_date": "2026-02-01",
        "delivery_date": None,
    },
    "ORD-1003": {
        "customer_email": "bob@example.com",
        "items": [{"name": "Mechanical Keyboard", "qty": 1, "price": 149.99}],
        "total": 149.99,
        "status": "delivered",
        "order_date": "2025-11-10",
        "delivery_date": "2025-11-15",
    },
}

PRODUCTS = {
    "SKU-001": {"name": "Wireless Headphones", "price": 79.99, "stock": 45},
    "SKU-002": {"name": "USB-C Cable", "price": 12.99, "stock": 200},
    "SKU-003": {"name": "Phone Case", "price": 24.99, "stock": 0},
    "SKU-004": {"name": "Mechanical Keyboard", "price": 149.99, "stock": 12},
    "SKU-005": {"name": "Monitor Stand", "price": 59.99, "stock": 30},
}

FAQ_ENTRIES = [
    {"question": "What is your return policy?", "answer": "Items can be returned within 30 days of delivery for a full refund. Items must be in original packaging."},
    {"question": "How long does shipping take?", "answer": "Standard shipping takes 3-5 business days. Express shipping takes 1-2 business days."},
    {"question": "Do you offer international shipping?", "answer": "Yes, we ship to over 50 countries. International shipping takes 7-14 business days."},
    {"question": "How do I track my order?", "answer": "You can track your order using the order ID on our website, or contact support with your order number."},
    {"question": "What payment methods do you accept?", "answer": "We accept Visa, Mastercard, AMEX, PayPal, and Apple Pay."},
]

RETURN_LOG = []  # Track processed returns

print("Backend data loaded: 3 orders, 5 products, 5 FAQ entries")

In [ ]:
# ============================================================
# Tool 1: Order Lookup
# ============================================================


@support_server.tool
def lookup_order(order_id: str) -> dict:
    """Look up an order by its order ID (e.g., ORD-1001). Returns order details including items, status, and dates."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return {"error": f"Order '{order_id}' not found. Please verify the order ID."}
    return {"order_id": order_id.upper(), **order}


# ============================================================
# Tool 2: Search Orders by Email
# ============================================================


@support_server.tool
def search_orders_by_email(email: str) -> list[dict]:
    """Find all orders associated with a customer email address."""
    results = []
    for order_id, order in ORDERS.items():
        if order["customer_email"].lower() == email.lower():
            results.append({"order_id": order_id, "total": order["total"], "status": order["status"]})
    if not results:
        return [{"message": f"No orders found for {email}"}]
    return results


# ============================================================
# Tool 3: Check Product Availability
# ============================================================


@support_server.tool
def check_product_availability(product_name: str) -> list[dict]:
    """Check if a product is in stock. Searches by product name (partial match supported)."""
    results = []
    for sku, product in PRODUCTS.items():
        if product_name.lower() in product["name"].lower():
            results.append({
                "sku": sku,
                "name": product["name"],
                "price": product["price"],
                "in_stock": product["stock"] > 0,
                "quantity_available": product["stock"],
            })
    if not results:
        return [{"message": f"No products found matching '{product_name}'"}]
    return results


# ============================================================
# Tool 4: Process Return Request
# ============================================================


@support_server.tool
def request_return(order_id: str, reason: str) -> dict:
    """Submit a return request for an order. Only delivered orders within 30 days of delivery are eligible."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return {"error": f"Order '{order_id}' not found."}

    if order["status"] != "delivered":
        return {"error": f"Order {order_id} has status '{order['status']}'. Only delivered orders can be returned."}

    if order["delivery_date"]:
        delivery = datetime.strptime(order["delivery_date"], "%Y-%m-%d")
        if (datetime.now() - delivery).days > 30:
            return {"error": f"Order {order_id} was delivered on {order['delivery_date']}, which is more than 30 days ago. Not eligible for return."}

    return_entry = {
        "return_id": f"RET-{len(RETURN_LOG) + 2001}",
        "order_id": order_id.upper(),
        "reason": reason,
        "refund_amount": order["total"],
        "status": "approved",
    }
    RETURN_LOG.append(return_entry)
    return return_entry


# ============================================================
# Tool 5: Search FAQ
# ============================================================


@support_server.tool
def search_faq(query: str) -> list[dict]:
    """Search the FAQ knowledge base for answers to common questions."""
    results = []
    query_lower = query.lower()
    for entry in FAQ_ENTRIES:
        if any(word in entry["question"].lower() or word in entry["answer"].lower() for word in query_lower.split()):
            results.append(entry)
    if not results:
        return [{"message": "No FAQ entries found. Please contact a human agent for assistance."}]
    return results


print(f"Customer Support MCP server defined with 5 tools!")

## Part 2: Testing Locally

Before connecting to OpenAI, let's verify all tools work using the in-memory client.

In [ ]:
from fastmcp import Client

async with Client(support_server) as c:
    tools = await c.list_tools()
    print("Tools available:")
    for t in tools:
        print(f"  - {t.name}: {t.description}")
    print()

    # Test: look up an order
    result = await c.call_tool("lookup_order", {"order_id": "ORD-1001"})
    print("Order lookup:", result[0].text)
    print()

    # Test: search by email
    result = await c.call_tool("search_orders_by_email", {"email": "alice@example.com"})
    print("Orders for alice:", result[0].text)
    print()

    # Test: check product stock
    result = await c.call_tool("check_product_availability", {"product_name": "keyboard"})
    print("Keyboard stock:", result[0].text)
    print()

    # Test: search FAQ
    result = await c.call_tool("search_faq", {"query": "return policy"})
    print("FAQ result:", result[0].text)

## Part 3: Deploy and Connect to OpenAI

Now let's run the server over HTTP, tunnel it with ngrok, and let GPT handle customer conversations.

In [ ]:
from pyngrok import ngrok

# Start the HTTP server
server_thread = threading.Thread(
    target=lambda: support_server.run(transport="http", host="127.0.0.1", port=8001),
    daemon=True,
)
server_thread.start()
time.sleep(2)

# Create ngrok tunnel
if not os.environ.get("NGROK_AUTHTOKEN"):
    ngrok_token = getpass("Enter your ngrok auth token: ")
    ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(8001)
mcp_url = f"{public_url}/mcp/"

print(f"Customer Support MCP server available at: {mcp_url}")

In [ ]:
# Helper function for our support conversations
def ask_support(question, previous_id=None):
    """Send a question to the AI customer support agent."""
    response = client.responses.create(
        model=MODEL,
        instructions="""You are a helpful customer support agent for an electronics store.
Be friendly, professional, and concise. Use the available tools to look up
real order and product information. Never make up order details — always
use the tools to get accurate data.""",
        tools=[
            {
                "type": "mcp",
                "server_label": "support",
                "server_url": mcp_url,
                "require_approval": "never",
            }
        ],
        input=question,
        previous_response_id=previous_id,
    )
    return response

### Scenario 1: Order Status Inquiry

In [ ]:
response = ask_support("Hi, I'd like to check the status of my order ORD-1002.")
print(response.output_text)

### Scenario 2: Finding All Orders for a Customer

In [ ]:
response = ask_support("Can you find all orders for alice@example.com?")
print(response.output_text)

### Scenario 3: Product Availability Check

In [ ]:
response = ask_support("Do you have any phone cases in stock? And what about mechanical keyboards?")
print(response.output_text)

### Scenario 4: Return Request

This demonstrates the model using business logic — the return tool validates eligibility (delivered status, within 30 days) before approving.

In [ ]:
# This should succeed — ORD-1001 was delivered recently
response = ask_support(
    "I'd like to return my order ORD-1001. The headphones don't fit comfortably."
)
print(response.output_text)

In [ ]:
# This should fail — ORD-1003 was delivered more than 30 days ago
response = ask_support(
    "I want to return order ORD-1003. The keyboard stopped working."
)
print(response.output_text)

### Scenario 5: FAQ Search

In [ ]:
response = ask_support("Do you ship internationally? And what payment methods do you accept?")
print(response.output_text)

### Scenario 6: Multi-Turn Conversation

Using `previous_response_id`, the model remembers the conversation context and can reference earlier information.

In [ ]:
# Turn 1: Look up an order
r1 = ask_support("I have order ORD-1001. Can you tell me what I ordered?")
print("Agent:", r1.output_text)
print()

# Turn 2: Follow-up question using conversation context
r2 = ask_support(
    "Is that item still available if I wanted to buy another one?",
    previous_id=r1.id,
)
print("Agent:", r2.output_text)
print()

# Turn 3: Another follow-up
r3 = ask_support(
    "Great. And what's your shipping policy for that?",
    previous_id=r2.id,
)
print("Agent:", r3.output_text)

## Part 4: Inspecting the MCP Interaction

Let's look at exactly what happens when OpenAI interacts with our MCP server.

In [ ]:
response = ask_support("Check order ORD-1002 and tell me if the phone case is still in stock.")

print("=== MCP Interaction Trace ===")
for i, item in enumerate(response.output):
    print(f"\nStep {i + 1} — {item.type}")
    if item.type == "mcp_list_tools":
        print(f"  Discovered {len(item.tools)} tools from '{item.server_label}' server")
    elif item.type == "mcp_tool_call":
        print(f"  Called: {item.name}({item.arguments})")
    elif item.type == "mcp_tool_call_output":
        output_preview = item.output[:100] + "..." if len(item.output) > 100 else item.output
        print(f"  Result: {output_preview}")
    elif item.type == "message":
        print(f"  Final answer: {response.output_text[:150]}...")

## Cleanup

In [ ]:
ngrok.disconnect(public_url.public_url)
print("ngrok tunnel closed.")

## From Notebook to Production

To deploy this MCP server in production, you would:

1. **Save the server as a Python file** — extract the `FastMCP` server code into `server.py`
2. **Replace simulated data with real backends** — connect to your actual database, APIs, etc.
3. **Deploy to a cloud host** — use any platform that runs Python (Railway, Fly.io, AWS, etc.)
4. **Add authentication** — pass headers in the OpenAI MCP config:

```python
{
    "type": "mcp",
    "server_label": "support",
    "server_url": "https://your-server.com/mcp/",
    "require_approval": "never",
    "headers": {
        "Authorization": "Bearer YOUR_API_KEY"
    }
}
```

The same MCP server also works with **Claude Desktop**, **VS Code**, **Cursor**, and any other MCP-compatible client — build once, connect everywhere.

## Summary

In this project, we built a complete customer support system powered by MCP:

- **5 tools** covering order lookup, product availability, returns processing, and FAQ search — all defined with simple `@server.tool` decorators.
- **Business logic validation** in tools (e.g., the return tool checks delivery date and order status) — the AI model respects these constraints.
- **OpenAI integration** via the Responses API — GPT automatically discovers and uses your tools to answer customer questions.
- **Multi-turn conversations** with `previous_response_id` for natural, context-aware support interactions.
- **Inspection** of the full MCP interaction trace showing tool discovery, calls, and results.

This pattern — wrapping your backend in an MCP server and connecting it to an AI model — is how companies are building AI-powered applications today. The MCP standard means your tools work with any compatible AI client, not just OpenAI.